# The optimization workflow

**Goal.** Run an optimization with APEX from start to finish: copy an example, read its configuration, validate it, submit it, wait for it and read the result. You will run two optimizers on Rosenbrock's function, edit a copy of a configuration and rerun it, then run two global-search optimizers on Rastrigin's function, where the shape of the problem matters. At the end the same work is looked at from the terminal.

**Prerequisites.** APEX installed and basic Python. The examples use analytic test functions, so there is no simulator to set up.

**Where files go.** The notebook uses APEX's Python client on a *services root* of its own: a folder where APEX keeps its jobs and results. Here it is `apex_home`, and example copies go in `work`. Both are relative to the notebook's working folder. (When no services root is given, the APEX commands that take `--root` use `$APEX_HOME`, and if that is not set, `~/.apex/service`. This notebook passes its own folder, so it touches neither.) For the terminal commands near the end, open a terminal and `cd` into the same folder, the one that will contain `work` and `apex_home` once the cells have run.

**Two kinds of state.** The *job* state (from `client.wait`) says whether the submitted job ran to its end: `complete` means it did. The *run* state (in the optimization summary) is the optimization's own state: `finished` means the optimizer stopped. Neither says the minimum was found; only the objective value does.

In [1]:
from pathlib import Path

# Relative paths below (work, apex_home) are resolved from this folder.
print("working folder name:", Path.cwd().name)
print("contents:", sorted(p.name for p in Path.cwd().iterdir()))

working folder name: optimization-e894b2ef
contents: []


## Step 1: copy an example from the catalogue

APEX ships a catalogue of examples. Copying one into a working folder gives you configuration files you can read and edit.

The example `optimization` holds two configurations of the same problem, *Rosenbrock's function minimised by EnOpt and by a trust region*: `rosenbrock_enopt.toml` and `rosenbrock_trust_region.toml`.

In [2]:
import pandas as pd

from apex.client import Client
from apex.services.examples import copy_example, examples_root, find_examples
from apex.services.settings import ExamplesSection

catalogue = {e.id: e for e in find_examples(examples_root(ExamplesSection()))}
copied = copy_example(catalogue["optimization"], Path("work"))
config = copied.folder / "rosenbrock_enopt.toml"
config_trust = copied.folder / "rosenbrock_trust_region.toml"
print(config.relative_to(Path.cwd()))
print(config_trust.relative_to(Path.cwd()))

work/optimization/rosenbrock_enopt.toml
work/optimization/rosenbrock_trust_region.toml


## Step 2: read the configuration

An optimization is described by one TOML file: the objective, the controls (the variables being optimized) and the optimizer with its options. Print the EnOpt file without its comments to see the settings alone.

In [3]:
def setting(line):
    for index, char in enumerate(line):
        if char == "#" and line[:index].count('"') % 2 == 0:
            return line[:index].rstrip()
    return line.rstrip()


settings = [setting(line) for line in config.read_text().splitlines()]
print("\n".join(line for line in settings if line))

[optimization]
name = "rosenbrock_enopt"
seed = 2026
[[controls.variables]]
name = "x"
initial = [-1.2, 1.0, -0.5, 0.8]
lower = -2.0
upper = 2.0
variance = 0.001
[objective]
name = "rosenbrock"
[optimizer]
name = "enopt"
options = { ne = 20, maxiter = 300, alpha = 0.05, optimizer = "Adam", beta = 0.9, resample = 2 }


How to read it:

- `[optimization]` gives the run a `name` and a `seed`, so a run can be repeated.
- `[[controls.variables]]` declares a variable `x`. Its `initial` list has four values, so the problem has four controls, which the results name `x[0]` to `x[3]`. The other keys of the block (`lower`, `upper`, `variance`) are described in the configuration reference.
- `[objective]` names the function to minimise, here `rosenbrock`.
- `[optimizer]` names the method and passes `options`. For `enopt` the options include `ne`, `maxiter` and `alpha` (with defaults of 100, 100 and 0.1); the EnOpt reference describes each one, along with `optimizer`, `beta` and `resample`. In this file `maxiter` is the iteration limit the optimizer works to.

With four controls, Rosenbrock's minimum is at `x = (1, 1, 1, 1)`, where the objective is 0.

Now the trust-region file. The objective and controls are the same; the optimizer section differs. Not every optimizer takes the same options, so check the reference for the one you choose.

In [4]:
settings_trust = [setting(line) for line in config_trust.read_text().splitlines()]
print("\n".join(line for line in settings_trust if line))

[optimization]
name = "rosenbrock_trust_region"
seed = 1
[[controls.variables]]
name = "x"
initial = [-1.2, 1.0, -0.5, 0.8]
[objective]
name = "rosenbrock"
[optimizer]
name = "trust_region"
options = { maxiter = 100 }


## Step 3: validate before running

Validation checks the configuration and reports a plan, without simulating anything. It is cheap, so make it a habit: a typo in a name shows up here at once instead of after a long run.

The plan's `max_evaluations` is the budget of objective calls the plan states. For some optimizers it is `None`, meaning the plan gives no figure. The helper below says so explicitly.

In [5]:
client = Client(Path("apex_home"))  # a services root of its own


def describe(plan):
    budget = plan.max_evaluations
    shown = "no fixed maximum in the plan" if budget is None else f"at most {budget} evaluations"
    print(f"{plan.optimizer} on {plan.objective}: {plan.n_controls} controls, {shown}")


checked = client.validate_optimization(config)
print("valid:", checked.valid)
plan = checked.optimization
describe(plan)

valid: True
enopt on rosenbrock: 4 controls, at most 23402 evaluations


## Step 4: submit and wait

`client.submit_optimization` takes the configuration and returns a handle that carries a job id. `client.wait` waits on that job, up to the timeout, and returns its status.

What to do with the outcome:

- `complete`: the job ran to its end; go on to read the result.
- Still running when the wait ends (a timeout): the job is not lost. Wait again with `client.wait(handle.id, ...)` and a longer timeout.
- Anything else, such as a failure: the code below prints the whole status, which carries the details, and then looks for a run folder with the job's id among `client.optimizations()`. If a row is found, its `path` is the run folder; read what is in it before changing the configuration. If no row is found, no run was created, and the status is all there is to read (a validation step like step 3 usually catches such a cause early).

In [6]:
handle = client.submit_optimization(config)
status = client.wait(handle.id, timeout_s=600, poll_s=0.5)
print("job state:", status.state)
if status.state != "complete":
    print(status)
    rows = [r for r in client.optimizations() if r["job"] == handle.id]
    if rows:
        print("run folder:", rows[0]["path"])
    else:
        print("no run folder was created for this job")

job state: complete


## Step 5: read the result

Find the run of our job by its job id among `client.optimizations()`, and ask for its summary. It holds the run's state, the final objective value and the controls found. The run state `finished` tells you the optimizer stopped; it does not tell you the minimum was reached, so always look at the objective.

In [7]:
row = next(r for r in client.optimizations() if r["job"] == handle.id)
summary = client.optimization_summary(Path(row["path"]))
print("run state:", summary.state, "|", summary.optimizer, "on", summary.objective)
print("objective:", summary.result["objective"])
names, values = summary.result["names"], summary.result["x"]
print("controls:", dict(zip(names, values, strict=True)))
print("run folder:", Path(row["path"]))

run state: finished | enopt on rosenbrock
objective: 0.9878013710933521
controls: {'x[0]': 0.6878032981993977, 'x[1]': 0.48472813220308203, 'x[2]': 0.26026533786111106, 'x[3]': 0.0680490417549587}
run folder: apex_home/optimizations/rosenbrock_enopt/b9e4a4a6d56b


Compare with the known answer: the minimum is 0 at `(1, 1, 1, 1)`. If the objective is clearly above 0, the run stopped short of the minimum. Rosenbrock's function has a long, curved, flat valley that is slow to follow, so stopping short is common and is not a sign that something went wrong.

The run folder has the form `apex_home/optimizations/<name>/<id>`; the id differs from run to run.

## Step 6: edit a copy and rerun

To try to get closer, give EnOpt a larger iteration limit, `maxiter`. Keep every run separate: write a copy of the file with a new `name` and a larger `maxiter`, then validate and submit the copy exactly as before. (A run already has its own directory, so do not reuse a name; this is also why the terminal section uses a copy.)

The code compares the two objectives itself, so you do not have to assume the outcome.

In [8]:
text = config.read_text()
long_text = text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_long"').replace(
    "maxiter = 300", "maxiter = 1000"
)
config_long = copied.folder / "rosenbrock_enopt_long.toml"
config_long.write_text(long_text)

# A second copy, kept for the terminal section below (same settings, its own name).
config_cli = copied.folder / "rosenbrock_enopt_cli.toml"
config_cli.write_text(text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_cli"'))

print("changed:", long_text != text)
checked_long = client.validate_optimization(config_long)
print("valid:", checked_long.valid)
describe(checked_long.optimization)

handle_long = client.submit_optimization(config_long)
status_long = client.wait(handle_long.id, timeout_s=900, poll_s=0.5)
print("job state:", status_long.state)
row_long = next(r for r in client.optimizations() if r["job"] == handle_long.id)
summary_long = client.optimization_summary(Path(row_long["path"]))
before = summary.result["objective"]
after = summary_long.result["objective"]
print("run state:", summary_long.state, "objective:", after)
if after < before:
    print(f"maxiter 1000 improved the objective: {before:.6f} -> {after:.6f}")
elif after == before:
    print("identical objective: the larger limit changed nothing")
else:
    print(f"the objective did not improve: {before:.6f} -> {after:.6f}")

changed: True
valid: True
enopt on rosenbrock: 4 controls, at most 78002 evaluations


job state: complete
run state: finished objective: 0.9878013710933521
identical objective: the larger limit changed nothing


Read the last line. If the objective improved, the extra iterations were used. If it is identical to step 5, the larger `maxiter` made no difference: the first run ended before reaching its limit, because one of the optimizer's own stopping rules fired (see the optimizer's reference for its tolerances such as `ftol`, `xtol` and `gtol`). The state `finished` does not say which rule it was. To see how the objective moved step by step, run `apex summary` on the run (see the terminal section). In that case a bigger iteration limit is not the way forward; changing other options (for example `alpha`, `ne` or the `seed`) in a renamed copy is.

## Step 7: run the other optimizer and compare

The same steps work for the trust-region configuration.

In [9]:
checked_trust = client.validate_optimization(config_trust)
print("valid:", checked_trust.valid)
describe(checked_trust.optimization)

handle_trust = client.submit_optimization(config_trust)
status_trust = client.wait(handle_trust.id, timeout_s=600, poll_s=0.5)
print("job state:", status_trust.state)

row_trust = next(r for r in client.optimizations() if r["job"] == handle_trust.id)
summary_trust = client.optimization_summary(Path(row_trust["path"]))

compare = pd.DataFrame(
    [
        {
            "optimizer": s.optimizer,
            "run": label,
            "state": s.state,
            "objective": s.result["objective"],
            "controls": [round(float(v), 3) for v in s.result["x"]],
        }
        for label, s in (
            ("maxiter 300", summary),
            ("maxiter 1000", summary_long),
            ("maxiter 100", summary_trust),
        )
    ]
)
print(compare.to_string(index=False))

valid: True
trust_region on rosenbrock: 4 controls, at most 502 evaluations


job state: complete
   optimizer          run    state  objective                      controls
       enopt  maxiter 300 finished   0.987801   [0.688, 0.485, 0.26, 0.068]
       enopt maxiter 1000 finished   0.987801   [0.688, 0.485, 0.26, 0.068]
trust_region  maxiter 100 finished   3.701429 [-0.776, 0.613, 0.382, 0.146]


Reading the table:

- A plan of `no fixed maximum` (the plan's value is `None`) means the plan does not state a number of evaluations for that optimizer, so the budgets cannot be compared in evaluations from the plan alone.
- What you can compare is the objective each run reached under its own stopping rule. A lower objective is closer to the minimum 0. The `state` column says only that the run ended.
- The conclusion is modest: the table shows how far each configuration got, not which method is better in general. The runs use different seeds and iteration limits, and a single run of a random method says little. For a fair comparison, give the methods the same budget measured the same way and repeat with several seeds.

## Step 8: when the landscape has many minima

Rosenbrock's function can trap a run in a local minimum: an objective clearly above 0 with controls away from `(1, 1, 1, 1)`. Other problems have far more. The `toy/rastrigin` example has a local minimum near every integer point: an ensemble gradient stops at the nearest one, while the population optimizers (the genetic algorithm and particle swarm) can find the global one. Its configurations are `rastrigin_ga.toml` and `rastrigin_pso.toml`. The workflow is the same, so we wrap the steps in a small function and run both.

In [10]:
rastrigin = copy_example(catalogue["toy/rastrigin"], Path("work"))


def optimize(cfg):
    """Validate, submit and wait for one configuration; return its plan and summary."""
    check = client.validate_optimization(cfg)
    print("valid:", check.valid)
    job = client.submit_optimization(cfg)
    final = client.wait(job.id, timeout_s=600, poll_s=0.5)
    print("job:", final.state)
    found = next(r for r in client.optimizations() if r["job"] == job.id)
    return check.optimization, client.optimization_summary(Path(found["path"]))


config_ga = rastrigin.folder / "rastrigin_ga.toml"
config_pso = rastrigin.folder / "rastrigin_pso.toml"
plan_ga, summary_ga = optimize(config_ga)
plan_pso, summary_pso = optimize(config_pso)

results = pd.DataFrame(
    [
        {
            "optimizer": p.optimizer,
            "max evaluations": p.max_evaluations,
            "objective": s.result["objective"],
            "controls": [round(float(v), 4) for v in s.result["x"]],
        }
        for p, s in ((plan_ga, summary_ga), (plan_pso, summary_pso))
    ]
)
print(results.to_string(index=False))

valid: True


job: complete
valid: True


job: complete
optimizer  max evaluations  objective         controls
       ga             4020   0.001114 [-0.0022, 0.001]
      pso             4020   0.000000      [-0.0, 0.0]


The global minimum of Rastrigin is 0 at the origin. An objective very close to 0 with controls close to `(0, 0)` means the global minimum was found. A run trapped in a local minimum would end near another integer point with an objective of about 1 or more. If both rows are near 0, a small difference between them is the precision each reached, not a different basin, and one run each is too little to rank the two methods.

To try another optimizer on a problem, change only the configuration file you pass to `optimize`.

## The same from the terminal

Do this after every cell above has run, in a terminal opened in the notebook's folder (the one holding `work` and `apex_home`).

**Where the runs go.** The notebook submitted its runs as jobs of the services root `apex_home`, so they sit in `apex_home/optimizations/<name>/<id>`.

**Start an optimization.** `apex optimize CONFIG` starts an optimization with one of PET's optimizers. `--no-progress` turns off the progress shown on standard error. `--max-iterations N` stops the run after N accepted iterations and the run can be resumed; it is separate from `maxiter` in the configuration, which is the optimizer's own iteration limit.

This runs the copy the notebook wrote in step 6 but did not run (it has its own `name`), pausing after 5 accepted iterations:

```bash
apex optimize work/optimization/rosenbrock_enopt_cli.toml --max-iterations 5 --no-progress
```

**Continue it.** `--resume` takes the directory of the run to continue. For a run started from `work/optimization/rosenbrock_enopt_cli.toml`, the run is in `work/optimization/runs/rosenbrock_enopt_cli`:

```bash
apex optimize --resume work/optimization/runs/rosenbrock_enopt_cli --max-iterations 5 --no-progress
```

**Summarise runs.** `apex summary` lists every metric at every step of the runs it finds under the folders you give, as Markdown on the screen, or in a file (with a CSV next to it) with `--output`. For an optimization, the rows are the optimization steps; look at the objective from the first step to the last. A healthy run shows the objective falling quickly at first and then levelling off. If it is still falling at the last step, the run stopped early: continue it or raise `maxiter` in a renamed copy. If it is flat from the start or rises, change the step size (`alpha`) or the seed in a renamed copy. This reads the runs the notebook made through the client:

```bash
apex summary apex_home
```

**Report one run.** `apex report RUN_DIR` prints the data mismatch history and performance metrics of a run, and with `--plots` also writes figures (PNG and PDF) into `RUN_DIR/report` unless you pass `--output`. Step 5 printed the run folder of the EnOpt run. Here it is found by listing `apex_home/optimizations/rosenbrock_enopt`, one folder per run named by its id (there is one run of that name, so the first listed is the right one):

```bash
apex report "$(ls -d apex_home/optimizations/rosenbrock_enopt/* | head -n 1)" --plots
```

The terminal does not replace the Python steps; they do the same work. To run a configuration a notebook already ran, copy it and give the copy another `name` in `[optimization]`, as in step 6.

## Pitfalls

- **Skipping validation.** A wrong name is caught at once by `validate_optimization`, but only after a whole run if you submit directly.
- **Mixing up the two states.** `complete` is the job's state; `finished` is the optimization run's. Both say the work ended, neither says the minimum was found. Judge the objective against what you know of the problem.
- **Assuming more iterations always help.** If a run did not stop at its limit, raising the limit changes nothing, as step 6 checks.
- **Reading one run as proof.** Random methods give different results for different seeds; repeat with several seeds if it matters.
- **Comparing unequal budgets.** A plan can show `None` for the evaluations. Then compare the objectives reached under stated stopping rules, and say so.
- **Rerunning under the same name.** Copy the configuration and change `name`; to continue a run, use `apex resume` or `apex optimize --resume`.
- **Choosing a local method for a multimodal problem.** A method that follows a local gradient stops at the nearest minimum. Use a global optimizer, as in step 8.
- **Looking in the wrong place.** Find results by job id through `client.optimizations()`, or list `apex_home/optimizations/<name>`, not by guessing folder names.
- **Running the terminal from another folder.** The relative paths `work` and `apex_home` depend on the folder you are in.

## What to read next

The examples catalogue has more optimization problems that follow the same workflow, including a family of multi-objective test problems (`toy/multi_objective/...`) solved by NSGA-II, NSGA-III and MOEA/D, and two toy-reservoir examples (`toy/toy_reservoir_npv_water` and `toy/toy_reservoir_robust_npv`) that trade NPV against water injected, or against its spread over model realizations. Copy one with `copy_example`, read its TOML as in step 2, and follow the same steps.